# Individual Database Profiling Report: `TMP_DF9`


--- 
## 1. Introduction

This document serves as the standardized Data Profiling Report for the `TMP_DF9` database, a pivotal component of the Teotihuacan Mapping Project's (TMP) digital archive. This analysis is situated within Workflow 4 of Phase 1 of the Digital TMP project, a foundational stage dedicated to the systematic, quantitative evaluation of legacy database architectures. The core purpose of this report is to provide a comprehensive, deep-dive analysis of the `TMP_DF9` schema by visualizing and interpreting a suite of pre-computed metrics. By presenting this granular, empirical evidence in a structured and reproducible format, this report establishes a baseline understanding of a single database's structural complexity, data quality, and analytical performance.

This report is a visualization and interpretation layer for a standardized set of metrics generated by an automated data profiling pipeline, as defined in the Phase 1 project plan. It does not represent a live analysis but rather a reproducible summary of the database's state, ensuring that the evaluation is consistent with the analyses of five other legacy and benchmark databases examined in this phase. The analysis will proceed systematically, beginning with a high-level schema overview and complexity assessment, followed by detailed table-level analysis of storage and health metrics, a granular column-level examination of data types and content profiles, and a concluding evaluation of performance on canonical analytical queries. The findings presented herein will serve as foundational evidence for the high-level comparative analysis and, ultimately, for the final Phase 1 White Paper, which will present a formal, evidence-based recommendation for a strategic architectural redesign of the unified TMP database in Phase 2.


--- 
## 2. Background


### 2.1. Context and Motivation

The primary goal of Phase 1 of the Digital TMP project is to conduct a systematic and quantitative evaluation of four legacy databases (`TMP_DF8`, `TMP_DF9`, `TMP_DF10`, `TMP_REAN_DF2`) and two modern benchmark databases. As outlined in the project's architectural and planning documents, this evaluation is a foundational step designed to generate the empirical evidence required to inform a strategic architectural redesign in Phase 2. The existing legacy databases, developed over several decades, exhibit a range of structural complexities and performance characteristics that must be rigorously measured and compared before a new, unified schema can be designed.

This report, focused specifically on `TMP_DF9`, is one of six standardized analyses that provide the granular evidence needed for this high-level comparison. By applying a consistent suite of profiling metrics to each database, the project can move beyond anecdotal or theoretical assessments of their respective strengths and weaknesses. This systematic, data-driven approach is critical for justifying the project's final architectural recommendations based on quantitative evidence rather than purely on theoretical principles. The findings from this individual analysis, when synthesized with those from its counterparts, will form the basis of a defensible, evidence-based strategy for building a performant, usable, and maintainable unified database for future Teotihuacan research.


### 2.2. Data: The `TMP_DF9` Database

Data File 9 (`TMP_DF9`) represents a significant evolutionary step in the management of the Teotihuacan Mapping Project's electronic data, marking the project's transition into the era of PC-based relational databases and GIS. Developed primarily by Ian Robertson in the 1990s and enhanced under a 1999-2002 NSF grant, `TMP_DF9` was created by migrating its predecessor, the mainframe-based `DF8`, into more modern formats, initially Paradox and then Microsoft Access. This migration was not merely a format conversion; it involved a substantial effort in identifying and correcting errors present in the earlier files and, most importantly, implementing a formal relational database structure. A key improvement was the integration of `TMP_DF9`'s attribute data with a digitized map of the TMP survey tracts (MF2/MF3), enabling sophisticated GIS-supported spatial analyses for the first time.

The content of `TMP_DF9` largely mirrors that of its predecessor, `DF8`, containing records for 5,046 archaeological "cases" or sites. It encompasses approximately 300 data fields covering administrative details (personnel, dates), locational coordinates, descriptive observations (site condition, materials), artifact counts from the original 1960s-70s tabulations, and architectural interpretations. However, its structure is radically different from `DF8`. `TMP_DF9` was designed as a highly normalized relational database, a decision that led to extreme structural fragmentation. The schema is composed of **62 tables** in total. The data for each archaeological site is partitioned across **18 core data tables** (e.g., `location`, `description`, `lithicFlaked`) linked by the shared `SSN` primary key. Compounding this complexity are an additional **45 "Codes" tables**, which serve as lookups to translate numeric codes in the core tables into human-readable text descriptions. This design, while adhering to the principles of Third Normal Form (3NF) to minimize data redundancy, creates a complex web of relationships that is difficult to navigate and query.

Despite its structural improvements over `DF8`, `TMP_DF9` inherits some of its predecessor's critical legacy issues. Most notably, it retains the use of **sentinel values** (`-1`) to represent missing or inapplicable data instead of standard SQL `NULL`s. This non-standard practice, as documented in the *Cowgill, Robertson & Sload (2012)* guide, poses a significant data integrity risk by potentially corrupting statistical calculations if not explicitly filtered. Furthermore, the database perpetuates the "Total Counts Problem," where stored aggregate totals (e.g., `cerPhTot.totTzac`) do not always perfectly match the sum of their constituent parts from more detailed tables, a known issue stemming from the long and complex history of data entry and analysis.


### 2.3. Methods: Database Profiling Metrics

The analysis presented in this report is based on a standardized suite of pre-computed metrics generated by the `02_run_profiling_pipeline.py` script, as outlined in the Phase 1 project plan. This methodological approach ensures that the evaluation of `TMP_DF9` is both reproducible and directly comparable to the analyses of the other five databases under review. The metrics were calculated from a live PostgreSQL instance of the database and saved to disk as discrete JSON and CSV files. This notebook serves as the visualization and interpretation layer for this static, pre-computed data, rather than performing a live analysis. This ensures that the findings reported here are a stable, verifiable snapshot of the database's characteristics at the time the pipeline was executed.

The profiling pipeline gathers data across several distinct categories to provide a holistic assessment of the database. The report will present findings from each of these categories in sequence. **Schema-level metrics** provide a high-level overview, including aggregate object counts (e.g., table count) and total database size. **Table-level metrics** offer a more granular view of individual tables, assessing their size, row counts, and health indicators such as table bloat. **Column-level analysis** provides the deepest insights, examining both the structure (data types, nullability) and content (NULL value percentages, cardinality) of every column. Finally, this report presents custom **interoperability scores** designed to heuristically measure relational complexity, as well as **performance benchmarks** that measure query latency on a set of canonical analytical workloads. This standardized suite of metrics provides the quantitative foundation for the rigorous, evidence-based comparison across all Phase 1 databases.


### 2.4. Hypotheses

Given `TMP_DF9`'s documented design as a highly normalized relational database, this analysis proceeds from a specific, central hypothesis. The schema, comprising 62 distinct tables with an extensive network of foreign key relationships, represents a classic application of normalization theory intended to ensure data integrity and minimize redundancy. It is hypothesized that this extreme structural fragmentation, while theoretically sound for a transactional (OLTP) system, will impose a severe and unacceptable performance penalty on the join-intensive analytical (OLAP) queries required for archaeological research. We predict that the `TMP_DF9` schema will be the slowest of all legacy databases on complex join queries and will be orders of magnitude slower than the denormalized benchmark databases, thereby providing strong quantitative evidence that this architecture is ill-suited for the project's read-heavy analytical goals.

---
## 3. Setup and Configuration


In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import SVG, Markdown, display
from plotly.subplots import make_subplots

# --- CONFIGURATION ---------------------------------------------------
# SET THIS VARIABLE to the name of the database you want to analyze.
# e.g., 'TMP_DF8', 'TMP_DF9', 'tmp_benchmark_wide_numeric', etc.
DATABASE_NAME = "TMP_DF9"  # <--- CHANGE THIS
# ---------------------------------------------------------------------

# --- Path Definitions ---
# Use relative paths from the notebook's location in reports/individual_db_analysis/
METRICS_DIR = Path("../../outputs/metrics")
ERDS_DIR = Path("../../outputs/erds")

# --- Styling and Display Options ---
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


def display_header(title):
    display(Markdown(f"### {title}"))


def load_metric_file(metric_name, file_type="csv"):
    """Helper function to safely load a metric file."""
    file_path = METRICS_DIR / f"{DATABASE_NAME}_{metric_name}.{file_type}"
    if not file_path.exists():
        print(f"⚠️ WARNING: Metric file not found: {file_path.name}")
        return None
    if file_type == "csv":
        return pd.read_csv(file_path)
    elif file_type == "json":
        with open(file_path, "r") as f:
            return json.load(f)


print(f"✅ Setup complete. Analyzing database: '{DATABASE_NAME}'")
print(f"Metrics Directory: {METRICS_DIR}")
print(f"ERD Directory: {ERDS_DIR}")

✅ Setup complete. Analyzing database: 'TMP_DF9'
Metrics Directory: ..\..\outputs\metrics
ERD Directory: ..\..\outputs\erds


---
## 4. Data Loading


In [ ]:
# Load all metric files into variables
basic_metrics = load_metric_file("basic_metrics", "json")
schema_counts = load_metric_file("schema_counts", "json")
interop_metrics = load_metric_file("interop_metrics", "json")

# Load table metrics and convert to DataFrame
table_metrics_data = load_metric_file("table_metrics", "json")
table_metrics_df = pd.DataFrame(table_metrics_data) if table_metrics_data else None

# Load column structure and convert to DataFrame
column_structure_data = load_metric_file("column_structure", "json")
column_structure_df = (
    pd.DataFrame(column_structure_data) if column_structure_data else None
)

# Load column profiles and convert to DataFrame
column_profiles_data = load_metric_file("column_profiles", "json")
column_profiles_df = (
    pd.DataFrame(column_profiles_data) if column_profiles_data else None
)

# Performance benchmarks remain as CSV
performance_df = load_metric_file("performance_benchmarks")

print("✅ Data loading complete.")

✅ Data loading complete.


---
## 5. High-Level Overview & Schema Visualization


### 5.1. Data: Database and Schema-Level Metrics

This section presents a high-level, aggregate analysis of the `TMP_DF9` database, providing a "30,000-foot view" of its overall size, composition, and structural complexity. The metrics presented below are schema-wide statistics, sourced from the `TMP_DF9_basic_metrics.json`, `TMP_DF9_schema_counts.json`, and `TMP_DF9_interop_metrics.json` files generated by the profiling pipeline. These summary statistics offer a quantitative starting point for assessing the database's architecture before proceeding to more granular table and column-level analyses.

The summary table will present key metrics that quantify the database's scale and relational complexity. `table_count` is a direct measure of structural fragmentation, representing the total number of user-defined tables within the schema. `database_size_mb` quantifies the total disk space consumed by the database. The remaining metrics are custom heuristic scores designed to measure relational complexity: the **Join Dependency Index (JDI)** measures the density of formal foreign key relationships; the **Logical Interoperability Factor (LIF)** assesses the potential for *ad hoc* joins based on column name and data type similarity; and the **Normalization Factor (NF)** provides a composite score to estimate the overall degree of schema normalization.


### 5.2. Theory & Methods: Schema Complexity and Interoperability Metrics

The schema-level metrics presented in this section are derived from two sources: direct queries against the PostgreSQL `information_schema` catalog and a suite of custom heuristic scores designed to quantify architectural complexity. The combination of these methods provides a multi-faceted, quantitative assessment of the database schema.

**Schema Object Counts:** Fundamental metrics such as `table_count`, `view_count`, and `function_count` are derived from straightforward `COUNT` queries on the relevant tables within the `information_schema`. For example, the table count is obtained by querying `information_schema.tables` where the `table_schema` matches the target schema of the analysis. These direct counts provide a baseline measure of the number of discrete objects that comprise the database structure.

**Interoperability Metrics:** To move beyond simple counts, three custom heuristic metrics were developed to provide a more nuanced assessment of relational complexity. These are defined as follows:
*   **Join Dependency Index (JDI):** The JDI is a measure of relational complexity based on the density of defined foreign key relationships. It is calculated as `JDI = foreign_key_count / max_possible_foreign_keys`, where `max_possible_foreign_keys` is derived from the number of tables (`n`) as `n * (n - 1) / 2`. A JDI score closer to 1.0 indicates a schema with a dense web of explicit relationships, suggesting higher normalization, while a score closer to 0 indicates a schema with few formal relationships, typical of denormalized or fragmented designs.
*   **Logical Interoperability Factor (LIF):** The LIF is a heuristic designed to estimate the potential for *ad hoc* joins where formal foreign keys may not exist. It operates on the assumption that columns with similar names and data types across different tables are likely to represent the same logical entity and are thus joinable. The metric is calculated by counting the number of distinct column name/data type pairs that appear in more than one table, providing a rough measure of logical cohesion.
*   **Normalization Factor (NF):** The NF is a composite score that combines the `table_count` and the `JDI` into a single, normalized value between 0 and 1. This score provides a holistic heuristic for the degree of schema normalization or fragmentation. Higher values suggest a more normalized schema (many tables with dense relationships), while lower values indicate a simpler, denormalized, or fragmented structure. The primary value of these heuristic scores lies not in their absolute values, but in their utility for *relative comparison* across the different database schemas analyzed in Phase 1.


In [ ]:
display_header(f"Key Metrics for: {DATABASE_NAME}")

summary_data = {}
if basic_metrics:
    summary_data.update(basic_metrics)
if schema_counts:
    summary_data.update(schema_counts)
if interop_metrics:
    summary_data.update(interop_metrics)
if table_metrics_df is not None:
    summary_data["total_estimated_rows"] = int(table_metrics_df["row_estimate"].sum())

if summary_data:
    summary_series = pd.Series(summary_data).rename("Value").to_frame()
    display(summary_series)
else:
    print("No summary metrics available.")

### Key Metrics for: TMP_DF9

### 5.3. Results: Key Metrics for `TMP_DF9`

The high-level metrics for `TMP_DF9` immediately reveal an architecture of significant structural complexity. The database contains a total of **62 tables** and occupies **28 MB** of disk space. The most striking initial finding is the sheer number of tables, which is the highest among all legacy databases analyzed.

The interoperability metrics quantify this complexity further. The Join Dependency Index (JDI) is **0.0719**, the Logical Interoperability Factor (LIF) is **9**, and the composite Normalization Factor (NF) is **0.3503**. While the JDI appears relatively low, this is a mathematical artifact of the extremely high table count; the raw number of foreign keys is substantial, but spread across a vast number of potential relationships. The NF, a more holistic measure, is the highest of the legacy databases, confirming that `TMP_DF9` represents the most normalized and fragmented schema in the study.


### 5.4. Discussion: Initial Assessment of Schema Complexity

The quantitative schema-level metrics provide strong, unequivocal support for the initial hypothesis that `TMP_DF9` is a highly complex and fragmented database. The table count of 62 is exceptionally high for a dataset representing a single core entity (the archaeological site) and is direct evidence of an extreme application of normalization principles. The Normalization Factor (NF) of 0.3503, the highest among the legacy schemas, confirms this assessment. This structural fragmentation is a direct result of its documented design as a true relational database, intended to succeed the partitioned-file structure of its predecessor, `TMP_DF8`.

This initial assessment strongly suggests that the `TMP_DF9` schema, while perhaps theoretically sound from a data integrity perspective for a transactional system, is likely to be impractical for its intended analytical purpose. The high degree of fragmentation implies that even moderately complex analytical queries will require numerous, costly join operations to reassemble a complete record for a given site. This creates a significant structural burden that is hypothesized to have a direct, negative impact on both query performance and analytical usability. The schema represents a textbook example of over-normalization for a read-heavy, analytical environment.

### 5.5. Data & Methods: Entity-Relationship Diagram (ERD)

An Entity-Relationship Diagram (ERD) is a visual representation of a database schema that illustrates the entities (tables), their attributes (columns), and the relationships between them. It serves as a critical tool for understanding the logical structure of a database, revealing its degree of normalization, the nature of its data dependencies, and its overall architectural complexity.

The ERD presented in this report was not manually drawn but was generated through an automated process to ensure it provides a completely accurate and objective reflection of the live `TMP_DF9` database schema. The diagram was created by the `03_generate_erds.py` script, which uses the `sqlalchemy-schemadisplay` library to programmatically inspect the live database's metadata. This process automatically discovers all tables, columns, and foreign key constraints and uses the `graphviz` software toolkit to render a graphical representation of these objects and their relationships. This automated methodology guarantees that the ERD is a direct, empirical visualization of the database's actual structure, free from any idealization or interpretation.


In [ ]:
display_header(f"Full ERD for: {DATABASE_NAME}")

try:
    # Find the most recent ERD file for the database
    erd_files = sorted(ERDS_DIR.glob(f"{DATABASE_NAME}_full_ERD_*.svg"), reverse=True)
    if erd_files:
        display(SVG(erd_files[0]))
    else:
        print(f"❌ ERROR: Full ERD SVG file not found for '{DATABASE_NAME}'.")
except Exception as e:
    print(f"An error occurred while displaying the ERD: {e}")

### Full ERD for: TMP_DF9

### 5.6. Results & Discussion: Visualizing Relational Complexity

The full Entity-Relationship Diagram for `TMP_DF9` is a quintessential "spaghetti diagram," providing a stark and immediate visual testament to its extreme structural complexity. The diagram is saturated with a dense web of 62 tables and an overwhelming number of relationship lines, making it nearly illegible at a glance. The `location` table sits at the center, acting as the primary hub, from which dozens of core data tables (e.g., `lithicFlaked`, `cerVessel`, `condition`) and their associated "Codes" lookup tables radiate outwards and interconnect. This visual evidence powerfully corroborates the quantitative metrics, translating the abstract scores for `table_count` (62) and `nf` (0.3503) into a tangible representation of architectural intricacy.

This diagram is not merely a depiction of complexity; it is an argument in itself. It visually demonstrates the high cognitive load and significant query-writing effort required to work with this database. To retrieve a complete analytical record, a user or application would need to traverse this labyrinth of joins, linking numerous data and code tables. This contrasts sharply with the simple, single-table diagrams of the benchmark databases. The ERD thus serves as powerful, intuitive evidence supporting the Phase 1 White Paper's central thesis: the `TMP_DF9` architecture, while a theoretically rigorous application of normalization, results in a schema that is pragmatically unusable and computationally inefficient for its intended analytical purpose.


---
## 6. Table-Level Analysis


### 6.1. Data: Table-Level Metrics

This section transitions from the high-level schema overview to a more granular analysis of the individual tables within the `TMP_DF9` database. The data presented here are sourced from the `TMP_DF9_table_metrics.json` file, which contains a detailed statistical profile for each of the 62 tables. By examining these metrics, we can identify the largest and most data-rich tables, assess their general health, and understand how data is distributed across the highly fragmented schema.

The upcoming summary tables and charts will present four key metrics for each table. The `row_estimate` provides a statistical approximation of the number of rows, offering a quick measure of a table's scale. `column_count` indicates the width or number of attributes in each table. `total_size` reports the total disk space consumed by the table and all of its associated indexes, identifying which tables are the largest contributors to the database's storage footprint. Finally, `bloat_percent` is a critical database health indicator, quantifying the percentage of a table's file that consists of unused, reclaimable space, which can impact query performance.

### 6.2. Theory & Methods: Assessing Table Health and Size

The table-level metrics presented in this section are calculated using standard PostgreSQL functions and statistical queries that provide efficient and reliable information about table size and health. The methods used are designed to avoid performance-intensive operations while still yielding accurate assessments.

**Row Estimates:** The `row_estimate` for each table is not derived from an expensive `COUNT(*)` operation, which would require a full table scan. Instead, it is a highly efficient estimate sourced directly from the `pg_class.reltuples` column in PostgreSQL's internal statistics catalog. This value is updated by the `ANALYZE` command (and autovacuum daemon) and typically provides a very close approximation of the actual row count for static or infrequently updated tables, making it a standard and performant method for assessing table size.

**Table and Index Size:** The various size metrics (`table_size`, `index_size`, `total_size`) are calculated using built-in PostgreSQL functions such as `pg_relation_size()` and `pg_total_relation_size()`. These functions measure the actual disk space allocated to the table's data file (the "heap") and its associated indexes. The values are presented in a human-readable format (e.g., kB, MB) for easier interpretation.

**Table Bloat:** Table bloat refers to unused space that accumulates within a PostgreSQL table's data file due to the database's Multi-Version Concurrency Control (MVCC) implementation. When rows are updated (`UPDATE`) or deleted (`DELETE`), the old row versions are not immediately removed from the file; they are marked as "dead" and remain until a `VACUUM` process reclaims the space. `bloat_percent` is a key indicator of database health and is calculated using a standard community-provided SQL query that statistically estimates the amount of this dead space. High bloat percentages can negatively impact performance by increasing the number of disk pages that must be scanned to satisfy a query. It often suggests a need for database maintenance, such as running a `VACUUM FULL` operation or tuning autovacuum settings.


In [ ]:
display_header("Table Metrics Summary")

if table_metrics_df is not None and not table_metrics_df.empty:
    display(
        table_metrics_df.sort_values(
            by="row_estimate", ascending=False
        ).style.background_gradient(
            cmap="viridis", subset=["row_estimate", "bloat_percent"]
        )
    )
else:
    print("No table metrics data available.")

### Table Metrics Summary

### 6.3. Results: Table Metrics Summary for `TMP_DF9`

The table metrics summary for `TMP_DF9` reveals a distinct two-tiered structure in its data distribution. The first tier consists of the **18 core data tables**, all of which share a consistent row estimate of **5,050 rows**. This uniformity is a critical finding, confirming that these tables are thematically partitioned segments of a single logical dataset, each holding different attributes for the same set of 5,050 archaeological sites. The second tier is composed of the numerous, much smaller "Codes" tables, which function as simple lookups and have row counts corresponding to the number of codes they contain (e.g., `Codes_personnel` has an estimated -1 rows).

Among the core data tables, `cerVessel` stands out for its width, containing 69 columns of detailed ceramic vessel data. The `fieldWorkers` and `labAnalysts` tables have the highest row estimates at 14,768 and 10,585 respectively. This is because they are not structured with one row per site, but rather one row per worker-site assignment, representing a many-to-many relationship. These two tables also exhibit the highest bloat percentages, both at **85%**. The remaining core tables show moderate bloat, typically between 30% and 70%.


In [ ]:
display_header("Largest Tables by Total Size and Bloat")

if table_metrics_df is not None and not table_metrics_df.empty:
    # Convert pretty size string to bytes for sorting
    def size_to_bytes(s):
        if not isinstance(s, str):
            return 0
        num, unit = s.split()
        num = float(num)
        if "KB" in unit:
            return num * 1024
        if "MB" in unit:
            return num * 1024**2
        if "GB" in unit:
            return num * 1024**3
        return num

    df_copy = table_metrics_df.copy()
    df_copy["total_bytes"] = df_copy["total_size"].apply(size_to_bytes)
    df_copy["bloat_bytes_val"] = df_copy["bloat_bytes"]

    top_10_size = df_copy.nlargest(10, "total_bytes")
    top_10_bloat = df_copy.nlargest(10, "bloat_bytes_val")

    # Display tables
    display(Markdown("**Top 10 Tables by Total Size**"))
    display(
        top_10_size[["table_name", "total_size", "row_estimate"]].reset_index(drop=True)
    )

    display(Markdown("**Top 10 Tables by Bloat Size**"))
    display(
        top_10_bloat[
            ["table_name", "bloat_size", "bloat_percent", "row_estimate"]
        ].reset_index(drop=True)
    )

    # Create subplots
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Top 10 Tables by Total Size", "Top 10 Tables by Bloat Size"),
    )

    fig.add_trace(
        go.Bar(
            y=top_10_size["table_name"],
            x=top_10_size["total_bytes"],
            orientation="h",
            name="Total Size",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            y=top_10_bloat["table_name"],
            x=top_10_bloat["bloat_bytes_val"],
            orientation="h",
            name="Bloat Size",
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        title_text=f"Table Size Analysis for {DATABASE_NAME}",
        height=500,
        showlegend=False,
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(title_text="Size (Bytes)", row=1, col=1)
    fig.update_xaxes(title_text="Bloat (Bytes)", row=1, col=2)
    fig.show()
else:
    print("No table metrics data available for plotting.")

### Largest Tables by Total Size and Bloat

**Top 10 Tables by Total Size**

**Top 10 Tables by Bloat Size**

### 6.4. Results: Largest Tables by Size and Bloat

The analysis of the largest tables by total on-disk size is dominated by the core data tables that have the most columns or indexes. The `condition` and `archMaterial` tables are the largest, occupying 2.38 MB and 2.21 MB respectively, largely due to their numerous indexes. They are followed by `complexData` and `archInterp`, both around 1.7 MB. The `cerVessel` table, despite having the most columns, is smaller at 1.15 MB, indicating its indexes are less extensive.

The analysis of table bloat identifies the `fieldWorkers` and `labAnalysts` tables as having the most significant amount of wasted space, with 0.44 MB and 0.31 MB of bloat respectively, corresponding to an 85% bloat factor for both. This exceptionally high level of bloat suggests that these tables, which manage many-to-many relationships, may have undergone significant data churn (insertions, updates, deletions) during the database's lifecycle without subsequent, aggressive `VACUUM` maintenance. While other tables show moderate bloat, the condition of `fieldWorkers` and `labAnalysts` points to a potential area of database health concern or historical data volatility.

### 6.5. Discussion: Identifying Key Tables and Health Concerns

The table-level analysis provides definitive quantitative evidence for the architectural design of `TMP_DF9` as a highly normalized, partitioned system. The consistent 5,050-row count across all 18 core thematic tables is the most significant finding, proving they are fragments of a single logical entity. This structure necessitates joining these tables to assemble a complete record, a process that is hypothesized to be computationally expensive. The largest tables in the database are not large due to their row counts, but due to their width (many columns, like in `cerVessel`) or the proliferation of indexes (as seen in `archMaterial` and `condition`).

From a database health perspective, the analysis is mixed. The 45 small "Codes" tables show 100% bloat, an artifact of their small size and static nature that is of no practical concern. However, the 85% bloat in the `fieldWorkers` and `labAnalysts` tables is notable. While the total database size of 28 MB is small by modern standards, making the performance impact of this bloat likely minor, it does suggest that these particular tables, which handle the most complex relationships (many-to-many), were the most difficult to manage and populate historically. This finding may indicate that these tables were a source of data entry errors or required significant post-load corrections, reflecting the inherent difficulty of managing highly relational data. Overall, the table-level metrics reinforce the conclusion that the database's primary characteristic is its structural fragmentation.

---
## 7. Column-Level Analysis


### 7.1. Data Type Frequencies

#### 7.1.1. Data & Methods

This analysis examines the fundamental composition of the `TMP_DF9` schema by profiling the data types used across all of its columns. The data for this section is sourced from the `TMP_DF9_column_structure.json` file, which contains metadata for every column in the database, including its assigned PostgreSQL data type. The method involves aggregating this data to count the frequency of each distinct data type. The resulting distribution provides critical insight into the database's design philosophy, particularly regarding how different kinds of information (e.g., categorical, numeric, textual) are physically stored. This choice has direct implications for storage efficiency, data integrity, and analytical usability.


In [ ]:
display_header("Data Type Distribution")

if column_structure_df is not None:
    type_counts = column_structure_df["data_type"].value_counts().reset_index()
    type_counts.columns = ["data_type", "count"]

    # Calculate percentages
    type_counts["percentage"] = (
        type_counts["count"] / type_counts["count"].sum() * 100
    ).round(2)

    # Display comprehensive table
    display(Markdown("**Complete Data Type Distribution**"))
    display(type_counts.style.format({"percentage": "{:.2f}%"}))

    # Display summary statistics
    display(Markdown("**Data Type Summary**"))
    summary_stats = pd.DataFrame(
        {
            "Total Columns": [type_counts["count"].sum()],
            "Unique Data Types": [len(type_counts)],
            "Most Common Type": [
                f"{type_counts.iloc[0]['data_type']} ({type_counts.iloc[0]['count']} columns)"
            ],
            "Least Common Type": [
                f"{type_counts.iloc[-1]['data_type']} ({type_counts.iloc[-1]['count']} columns)"
            ],
        }
    )
    display(summary_stats)

    fig = px.bar(
        type_counts,
        x="data_type",
        y="count",
        title=f"Column Data Type Frequencies in {DATABASE_NAME}",
        labels={"count": "Number of Columns", "data_type": "Data Type"},
    )
    fig.show()
else:
    print("No column structure data available.")

### Data Type Distribution

**Complete Data Type Distribution**

**Data Type Summary**

,Total Columns,Unique Data Types,Most Common Type,Least Common Type
0,393,2,smallint (344 columns),text (49 columns)


#### 7.1.2. Results & Discussion

The analysis of column data types in `TMP_DF9` reveals a schema built almost exclusively on two types. Of the 393 total columns, **344 (87.53%)** are defined as `smallint` (a two-byte integer), and the remaining **49 columns (12.47%)** are `text`. There are no floating-point numbers, booleans, or other specialized data types used.

This distribution is a direct legacy of the design principles inherited from `DF8` and earlier mainframe systems, where integer codes were used to represent nearly all non-textual information. The `text` fields are used for descriptive identifiers like `site`, `subsite`, `unit`, and the `description` fields within the `Codes_` tables. Every other variable—from artifact counts to qualitative assessments of site condition—is stored as a `smallint`. This design choice, documented in the *Cowgill, Robertson & Sload (2012)* report and the Phase 1 White Paper, forces an external dependency on codebooks or the 45 separate `Codes_` tables to translate these integer codes into meaningful information. This severely impairs the database's usability for exploratory analysis and introduces the risk that these coded categorical variables might be incorrectly treated as continuous quantitative data in statistical analyses, a significant data integrity concern that a Phase 2 redesign must address.


### 7.2. Data Completeness: NULL Value Analysis

#### 7.2.1. Data & Methods

This analysis assesses data completeness and quality by measuring the prevalence of `NULL` values across every column in the `TMP_DF9` database. Sourced from the `TMP_DF9_column_profiles.json` file, the core metric is `null_percent`, which calculates the percentage of rows containing a `NULL` for each column. In standard SQL, `NULL` is the correct and conventional representation for missing or unknown data. A high percentage of `NULL` values in a column can indicate issues with the original data collection process or subsequent data entry, potentially impacting the reliability of any analysis that relies on that column. This analysis is therefore a critical measure of overall data quality and integrity.


In [ ]:
display_header("Top 20 Columns by Percentage of NULL Values")

if column_profiles_df is not None and not column_profiles_df.empty:
    # Ensure we only show columns with NULLs
    null_df = column_profiles_df[column_profiles_df["null_percent"] > 0].copy()

    if not null_df.empty:
        # Create a full column identifier for clarity
        null_df["full_column_name"] = (
            null_df["tablename"] + "." + null_df["column_name"]
        )

        top_20_nulls = null_df.nlargest(20, "null_percent")

        # Display table
        display(Markdown("**Top 20 Columns with Highest NULL Percentages**"))
        table_display = top_20_nulls[
            [
                "full_column_name",
                "null_percent",
                "null_count_estimate",
                "row_count_exact",
            ]
        ].copy()
        table_display.columns = ["Column", "NULL %", "NULL Count", "Total Rows"]
        display(table_display.reset_index(drop=True))

        # Display summary statistics
        display(Markdown("**NULL Value Summary**"))
        null_summary = pd.DataFrame(
            {
                "Total Columns Analyzed": [len(column_profiles_df)],
                "Columns with NULLs": [len(null_df)],
                "Columns with 100% NULLs": [
                    len(null_df[null_df["null_percent"] == 100])
                ],
                "Average NULL %": [f"{null_df['null_percent'].mean():.2f}%"],
                "Median NULL %": [f"{null_df['null_percent'].median():.2f}%"],
            }
        )
        display(null_summary)

        fig = px.bar(
            top_20_nulls,
            y="full_column_name",
            x="null_percent",
            orientation="h",
            title=f"Top 20 Columns by NULL Percentage in {DATABASE_NAME}",
            labels={
                "null_percent": "Percentage of Rows that are NULL (%)",
                "full_column_name": "Column",
            },
        )
        fig.update_layout(height=600)
        fig.update_yaxes(autorange="reversed")
        fig.show()
    else:
        print("✅ Excellent! No columns with NULL values were found.")

        # Still show summary even when no NULLs
        display(Markdown("**NULL Value Summary**"))
        null_summary = pd.DataFrame(
            {
                "Total Columns Analyzed": [len(column_profiles_df)],
                "Columns with NULLs": [0],
                "Data Completeness": ["100% - Perfect!"],
            }
        )
        display(null_summary)
else:
    print("No column profile data available.")

### Top 20 Columns by Percentage of NULL Values

✅ Excellent! No columns with NULL values were found.


**NULL Value Summary**

,Total Columns Analyzed,Columns with NULLs,Data Completeness
0,305,0,100% - Perfect!


#### 7.2.2. Results & Discussion

The NULL value analysis of `TMP_DF9` produces a result identical to its predecessor, `TMP_DF8`: across all 305 columns profiled in the core data tables, **zero `NULL` values were found**, resulting in a nominal data completeness score of 100%. This seemingly perfect result is, however, an artifact of a known legacy design choice rather than an indication of perfect data collection.

As documented in the historical project guides and confirmed in the `Cowgill, Robertson & Sload (2012)` report, `TMP_DF9` perpetuated the use of **sentinel values** to represent missing data. The integer `-1` is used throughout the database to signify "missing data" or "not applicable." Therefore, the 100% completeness score is misleading, as it masks a significant but unquantified amount of missing information encoded as a number. This non-standard practice carries severe implications for data integrity. Any statistical operation (e.g., calculating an average or sum) performed on a column containing these sentinel values will produce an incorrect result unless the user is aware of this "magic number" and explicitly filters it out. This design shifts the burden of managing missing data from the database system to the user, creating a significant risk of analytical error. This finding provides one of the strongest justifications for the Phase 2 redesign, a core requirement of which must be the systematic conversion of all sentinel values to standard SQL `NULL`s.


### 7.3. Data Complexity: Cardinality Analysis

#### 7.3.1. Data & Methods

This section analyzes the complexity of the data within each column by measuring its **cardinality**. The data is sourced from the `TMP_DF9_column_profiles.json` file. Cardinality is defined as the number of unique or distinct values present in a column. This metric is a fundamental characteristic of a dataset and is critical for understanding the nature of each attribute.

The analysis of cardinality helps to distinguish between different types of columns. Columns with very high cardinality, where nearly every value is unique, are typically identifiers or primary keys (e.g., `SSN`). Conversely, columns with very low cardinality (e.g., 2 to 10 distinct values) usually represent categorical variables, flags, or status codes (e.g., site condition codes, presence/absence flags). By examining the distribution of cardinalities across the schema, we can gain insight into the database's structure, identify potential join keys, and understand the complexity of the data that analysts will encounter.

In [ ]:
display_header("Column Cardinality Distribution")

if column_profiles_df is not None and not column_profiles_df.empty:
    # Create a full column identifier
    df = column_profiles_df.copy()
    df["full_column_name"] = df["tablename"] + "." + df["column_name"]

    # Display tables of highest and lowest cardinality columns
    display(Markdown("**Columns with Highest Cardinality (Most Unique)**"))
    display(
        df.nlargest(10, "distinct_values_estimate")[
            ["full_column_name", "distinct_values_estimate"]
        ]
    )

    display(Markdown("**Columns with Lowest Cardinality (Least Unique)**"))
    display(
        df[df["distinct_values_estimate"] > 1].nsmallest(
            10, "distinct_values_estimate"
        )[["full_column_name", "distinct_values_estimate"]]
    )

    # Create a histogram of cardinalities to see the distribution
    fig = px.histogram(
        df,
        x="distinct_values_estimate",
        log_y=True,
        title=f"Distribution of Column Cardinalities in {DATABASE_NAME}",
        labels={"distinct_values_estimate": "Number of Distinct Values (Cardinality)"},
    )
    fig.show()
else:
    print("No column profile data available.")

### Column Cardinality Distribution

**Columns with Highest Cardinality (Most Unique)**

**Columns with Lowest Cardinality (Least Unique)**

#### 7.3.2. Results & Discussion

The cardinality analysis of `TMP_DF9` clearly illustrates the structure of a highly relational and coded database. The columns with the highest cardinality are overwhelmingly identifiers or raw counts. The `location.SSN` column serves as the primary key with an estimated -1 distinct values (indicating all are unique), while raw artifact counts like `lithicFlaked.obsidianTot` (386 unique values) and `cerPhTot.totTlam` (233 unique values) also show high variability, as expected.

The most telling finding is the histogram of cardinalities, which is heavily skewed towards the low end. There is a very long tail of columns with extremely low cardinality. For instance, `cerVessel.comalPatl` has only 2 distinct values, `Codes_McomplexGen.presMcTl` has 4, and dozens of columns representing coded assessments of materials, site conditions, or architectural features (e.g., `description.slope`, `archMaterial.adobe`, `condition.dam`) have fewer than 10 unique values. This pattern is a direct, quantitative consequence of the database's design.

The vast number of low-cardinality columns confirms that most of the descriptive data in `TMP_DF9` is stored as integer codes that reference the `Codes_` lookup tables. Furthermore, it reinforces the "column-based artifact design" critique from the Phase 1 White Paper, a violation of First Normal Form (1NF). Many tables, such as `cerVessel`, contain dozens of separate columns for each artifact type (e.g., `ollaPatl`, `ollaMicc`, `comalTzac`), each of which is a low-cardinality count. This design makes aggregate queries (e.g., "what is the total olla count?") extremely cumbersome, requiring the manual summation of numerous columns. This cardinality profile provides strong quantitative evidence of an architecture that, while normalized, is profoundly inefficient for the types of aggregate analytical queries common in archaeological research.


---
## 8. Performance Benchmark Analysis


### 8.1. Data, Theory & Methods

This section evaluates the analytical performance of the `TMP_DF9` database by measuring query execution latency on a set of standardized, canonical queries. The data for this analysis is sourced from the `TMP_DF9_performance_benchmarks.csv` file, which logs the results of these benchmark tests. The methodology is designed to provide a fair, reproducible, and representative test of the schema's efficiency under different analytical workloads.

The methodology involves executing a predefined set of three canonical queries against the database and measuring the time taken for each to complete, reported in milliseconds. These queries are not generic; they are hand-crafted and stored in the `phases/01_LegacyDB/sql/canonical_queries/canonical_queries_df9.sql` file to specifically test the architectural characteristics of the `TMP_DF9` schema. The three queries represent distinct analytical workloads:
1.  **Baseline Scan:** A simple `COUNT(*)` on the main site table (`location`) to establish a baseline for raw I/O performance on a single, core table.
2.  **Multi-Table Join:** A query that joins five of the core data tables (`location`, `description`, `archInterp`, `lithicFlaked`, `admin`) to simulate a typical analytical task requiring a comprehensive site record.
3.  **Complex Filtering:** A query that joins three tables and applies filtering conditions on attributes from different tables, representing a more complex but common analytical scenario.

By comparing the latency of the join-intensive queries to the baseline, we can quantitatively measure the performance impact of `TMP_DF9`'s highly normalized design, directly testing the central hypothesis of this report.


In [ ]:
display_header("Canonical Query Performance Results")

if performance_df is not None and not performance_df.empty:
    display(performance_df[["query_name", "latency_ms", "status"]])

    # Plot the results for successful queries
    success_df = performance_df[performance_df["status"] == "Success"]
    if not success_df.empty:
        fig = px.bar(
            success_df,
            x="query_name",
            y="latency_ms",
            title=f"Query Latency for {DATABASE_NAME}",
            labels={"latency_ms": "Latency (ms)", "query_name": "Canonical Query"},
        )
        fig.show()
else:
    print("No performance benchmark data available.")

### Canonical Query Performance Results

### 8.2. Results: Query Performance for `TMP_DF9`

The performance benchmark results for `TMP_DF9` provide definitive, quantitative evidence of the severe performance cost associated with its highly normalized schema. The three canonical queries executed successfully, with the following recorded latencies:

*   **Baseline Performance - Query 1.1:** 1.55 ms
*   **Join Performance - Query 2.1:** 18.77 ms
*   **Complex Filtering - Query 3.1:** 2.60 ms

The baseline query, a simple count on a single table, executed very quickly at 1.55 ms. The join performance query, which required joining five core data tables to assemble a complete record, took **18.77 ms**. This represents a dramatic performance degradation of **12.1 times** compared to the baseline scan. While the complex filtering query was faster at 2.60 ms, it still represents a notable slowdown relative to the simple I/O test. The most significant finding is the extreme latency penalty incurred by the five-table join, which is a direct measure of the overhead imposed by the schema's fragmentation.


### 8.3. Discussion: Impact of Schema on Analytical Performance

The performance results provide powerful, quantitative validation for the central hypothesis of this analysis: the highly normalized, 62-table schema of `TMP_DF9` is ill-suited for the project's read-heavy analytical goals. The 12.1x performance degradation observed for the join-intensive query is a direct and predictable consequence of this architectural choice. To satisfy the query, the database engine must perform costly operations to link five of the 18 core data tables, each containing 5,050 rows, to assemble a complete record. This contrasts sharply with the near-instantaneous baseline query, which involves a simple scan of a single table.

This finding is critically important as it moves the argument against normalization from a theoretical discussion to an evidence-based conclusion. It demonstrates that the architectural decisions made to ensure data integrity in a transactional model come at an unacceptable performance cost for an analytical system. The need to perform such complex joins for routine analytical tasks creates a significant performance bottleneck that would be magnified in the final integrated geospatial database, where these queries would be combined with even more expensive spatial operations. This result provides the "smoking gun" evidence needed to justify the strategic move to a denormalized, wide-format architecture in Phase 2, which is designed specifically to eliminate this join-related overhead and ensure an efficient research environment.


---
## 9. Final Report and Recommendations


### 9.1. Summary of Results

This analysis provides a comprehensive, multi-faceted profile of the `TMP_DF9` legacy database, yielding a set of integrated findings that clearly characterize its architecture, data quality, and performance.

*   **Structural Complexity:**
    The relational structure of `TMP_DF9` is a textbook example of a highly normalized schema, characterized by extreme structural fragmentation. The database is partitioned into **62 distinct tables**, with 18 core data tables and 45 lookup "Codes" tables. This is visually represented in the ERD as a dense, complex "spaghetti diagram." The quantitative interoperability metrics confirm this visual assessment, with a Normalization Factor (NF) of **0.3503**, the highest among the legacy databases. The schema's composition is overwhelmingly dominated by the `smallint` data type (**87.5% of all columns**), reflecting a design philosophy that relies on numeric codes for categorical data, a practice that hinders direct usability and requires complex joins to the lookup tables for interpretation.

*   **Data Quality & Health Concerns:**
    `TMP_DF9` perpetuates a critical data quality issue from its predecessor, `DF8`: the use of **sentinel values (`-1`)** instead of standard SQL `NULL`s. The analysis found **zero NULL values**, a misleading indicator of completeness that masks missing data and poses a significant risk to the integrity of statistical calculations. In terms of database health, table bloat is a concern primarily for the many-to-many link tables, `fieldWorkers` and `labAnalysts`, which both exhibit **85% bloat**. While the performance impact is minor given the database's small total size (28 MB), this suggests these were the most historically volatile and difficult-to-manage tables.

*   **Performance Profile:**
    The performance benchmarks delivered the most decisive finding of this analysis. The query requiring a five-table join to assemble a complete analytical record ran **12.1 times slower** than the simple baseline scan of a single table (**18.77 ms** vs. **1.55 ms**). This dramatic performance degradation provides direct, quantitative evidence of the severe computational overhead imposed by the schema's high degree of normalization. The need to perform such complex and costly joins for routine analytical tasks represents a major architectural flaw for a read-heavy system.


### 9.2. Discussion

The most striking feature of the `TMP_DF9` database is its unwavering commitment to the principles of high normalization, a characteristic that defines its identity and is the root cause of its primary strengths and, more significantly, its profound weaknesses. The schema's primary characteristic is one of **extreme structural fragmentation in service of theoretical data integrity**. It is the logical successor to the partitioned-file design of `DF8`, taking the concept of separating data to its relational conclusion. This resulted in a schema with 62 tables, a formal network of foreign key constraints, and a near-total reliance on lookup tables—a design that, in a transactional (OLTP) context, would be lauded for minimizing data redundancy.

However, in the context of the Digital TMP project's analytical (OLAP) goals, this design is a critical flaw. The fragmentation that ensures write-time integrity becomes a massive performance and usability burden at read-time. The "spaghetti" ERD, the high Normalization Factor, the poor join performance, and the reliance on opaque `smallint` codes are all symptoms of this fundamental mismatch between architecture and application. The database is a well-executed example of an architectural pattern that is simply wrong for the problem it is intended to solve.

### 9.3. Conclusions & Implications for Phase 2 Redesign

Based on the comprehensive analysis of its structure, data quality, and performance, this report concludes that the `TMP_DF9` database, while a significant step in the evolution of the TMP's data management, is an unsuitable model for the final, unified database. Its architecture prioritizes theoretical normalization over the practical needs of analytical performance and user accessibility, creating critical bottlenecks that would be severely detrimental to the project's long-term goals.

*   **Based on this analysis, what are the key strengths and weaknesses of this database's design?**
    *   **Strengths:**
        *   **High Data Integrity (in theory):** The highly normalized design with foreign key constraints effectively prevents data modification anomalies (insertion, update, deletion), making it a robust model for a transactional system where data is frequently written and modified.
        *   **Storage Efficiency (for codes):** Storing categorical data as `smallint` codes rather than full text strings minimizes the on-disk storage footprint of the raw data.
    *   **Weaknesses:**
        *   **Severe Join Performance Penalty:** The extreme fragmentation imposes a crippling performance cost on analytical queries, as evidenced by the **12.1x slowdown** for a standard multi-table join. This is its single greatest weakness.
        *   **Poor Analytical Usability:** The schema is exceptionally difficult for end-users to work with. It requires writing complex, multi-table join queries and constant reference to lookup tables to interpret the data, creating a high barrier to entry for analysis.
        *   **Legacy Data Quality Issues:** It inherits the critical data integrity risk of using sentinel values (`-1`) for missing data instead of standard SQL `NULL`s.

*   **What specific aspects of this schema should be preserved, changed, or discarded in the final unified database?**
    *   **Discard:**
        *   **Extreme Normalization/Fragmentation:** The core architectural strategy of partitioning data across 18 core tables and 45 lookup tables must be completely discarded in favor of a denormalized model.
        *   **Sentinel Values for Missing Data:** The use of `-1` must be abandoned and these values systematically converted to standard SQL `NULL`s.
        *   **Numeric Coding for Categorical Data:** The reliance on `smallint` codes should be discarded.
    *   **Change:**
        *   **Data Types:** All columns representing categorical data must be changed from `smallint` to a descriptive `TEXT` type, with the human-readable string values stored directly in the primary table.
    *   **Preserve:**
        *   **Core Data Content and Variables:** The rich set of 300+ variables is the database's most valuable asset and must be preserved in its entirety.
        *   **The `SSN` Primary Key:** The concept of the `SSN` as the unique site identifier is sound and must be retained as the primary key in the new, unified structure.
        *   **Relational Logic (Implicitly):** The *logic* of the relationships defined by the foreign keys should be preserved, but it should be implemented by pre-joining the data into a single wide table rather than through a live, multi-table schema.
